In [3]:
import os
from langchain_cerebras import ChatCerebras
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from pymongo import MongoClient
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from langchain.agents import create_agent

load_dotenv()

CEREBRAS_API_KEY=os.getenv("CEREBRAS_API_KEY")
BREVO_API_KEY=os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")


In [4]:
DB_NAME = "soulsync"
COLLECTION_NAME="users"
SENDER_NAME="SoulSync"
SENDER_MAIL="lokeshvijayraina@gmail.com"


In [5]:
llm_gpt=ChatCerebras(model="gpt-oss-120b",api_key=CEREBRAS_API_KEY)
client= MongoClient(MONGODB_URI)
users_collection=client[DB_NAME][COLLECTION_NAME]


In [6]:
class EmailTemplate(BaseModel):
    subject_template:str
    body_template:str
    reason:str

class EmailDraft(BaseModel):
    receiver_mail:str
    subject:str
    body:str

class DispatchedUsers(BaseModel):
    totalUsers:str
    sent:str
    failed:str

In [ ]:
@tool
def find_users(
    filters: dict = None,
    sort_by: str = None,
    ascending: bool = False,
    limit: int = 5,
):
    """
    Query SoulSync users with optional filters, sorting, and limit.

    Available fields:
    - name
    - email
    - totalListeningTime
    - updatedAt (last active)
    - createdAt (joined date)
    """

    query = filters or {}

    targeted_users = users_collection.find(
        query,
        {
            "_id": 0,
            "name": 1,
            "email": 1,
            "totalListeningTime": 1,
            "updatedAt": 1,
            "createdAt": 1,
        },
    )

    if sort_by:
        targeted_users = targeted_users.sort(
            sort_by,
            -1 if not ascending else 1,
        )

    targeted_users = targeted_users.limit(limit)

    return list(targeted_users)

In [14]:
#find_users.invoke({"filters": {}, "sort_by": "totalListeningTime", "ascending": False, "limit": 5})


In [15]:
finder_agent=create_agent(
    model=llm_gpt,
    tools=[find_users],
    system_prompt=
    """
    You are LookOut's user discovery agent.

Your job is to identify the correct SoulSync users by calling the `find_users` tool.

Available fields:

* `name`
* `email`
* `country`
* `totalListeningTime` (seconds)
* `createdAt` (join date)
* `updatedAt` (last active)

Infer these arguments from the user's request:

* `filters`
* `sort_by`
* `ascending`
* `limit`

Intent examples:

* Top / most active → `sort_by="totalListeningTime"`, `ascending=False`
* Least active → `sort_by="totalListeningTime"`, `ascending=True`
* Newest → `sort_by="createdAt"`, `ascending=False`
* Oldest → `sort_by="createdAt"`, `ascending=True`
* Recently active → `sort_by="updatedAt"`, `ascending=False`
* Inactive → `sort_by="updatedAt"`, `ascending=True`
* Country-specific requests → use `filters`

Rules:

* Always call `find_users`.
* Never invent users or fields.
* Combine filters and sorting when needed.
* If the request cannot be answered using the available fields, explain why.

    """)

In [ ]:
response = finder_agent.invoke({
    "messages":[ HumanMessage(content="tell the latest active users today ")]
})


{'messages': [HumanMessage(content='tell the latest active users today ', additional_kwargs={}, response_metadata={}, id='82921d1e-5f39-4d7d-a44e-618114c8057b'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning': 'The user: "tell the latest active users today". They want recently active users (latest active). That maps to sort_by="updatedAt", ascending=False (most recent). Also maybe filter for today? No date filter field. Could filter by updatedAt? Probably not supported. So just sort by updatedAt descending and maybe limit default 10. Provide result.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 452, 'total_tokens': 555, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 73, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-oss-120b', 'system_fingerprint': 'fp_8a76e344b64d75413

In [27]:
print(response["messages"][-1].content)

Here are the users with the most‑recent activity (sorted by `updatedAt` descending). No one has an `updatedAt` timestamp for **today (2026‑07‑06)**, so the list shows the latest activity up to yesterday.

| # | Name | Email | Joined (`createdAt`) | Last active (`updatedAt`) | Total listening time (seconds) |
|---|------|-------|----------------------|---------------------------|---------------------------------|
| 1 | Jmvijay | jmvijay1@gmail.com | 2026‑07‑05 17:37:36 | 2026‑07‑05 19:31:21 | 10 446 |
| 2 | Vivi | abivino181131@gmail.com | 2026‑07‑05 17:51:17 | 2026‑07‑05 19:31:21 | 9 150 |
| 3 | Loki | lokeshvijayraina@gmail.com | 2026‑03‑04 08:44:49 | 2026‑07‑05 18:31:24 | 471 426 |
| 4 | Jai Bharath | jaibharath04102005@gmail.com | 2026‑03‑28 08:47:36 | 2026‑07‑05 16:39:30 | 147 662 |
| 5 | Sandy Dhanush | weortheboys1513@gmail.com | 2026‑05‑12 06:10:13 | 2026‑07‑05 16:38:41 | 156 977 |

These are the most recent active users based on the data available. If you need a different slice